# zagg query pipeline

Three AOIs, full ICESat-2 mission timeseries, with Sentinel-2 demo as well. Catalogs and shard maps land in `outputs/` (consumed by the write notebook).

In [1]:
# %pip install "zagg[catalog,viz]"

import logging
import os
import time
from datetime import date
from pathlib import Path

import boto3
import numpy as np

from zagg.catalog import load_polygon
from zagg.catalog.shardmap import ShardMap
from zagg.catalog.sources import Catalog, CMRSource, Query, STACQuery, STACSource
from zagg.config import default_config
from zagg.data import demo_aoi
from zagg.grids import HealpixGrid, from_config
from zagg.viz import show_shardmap

This next cell makes a best effort attempt at grabbing our un-merged spatial index PR on spherely, which is more performant and not sensative to grid size. Degradation without it is correct, but around 4x slower for large queries.

In [2]:
# Exact-S2 intersection backend: spherely fork wheels (SpatialIndex branch) from the
# demo bucket -- macOS arm64 + manylinux x86_64, cp310-cp314. Uses ambient AWS
# credentials (CryoCloud hub role / AWS profile). Without a matching wheel everything
# below still runs: ShardMap.build's backend="auto" falls back to mortie.
import importlib.util

if importlib.util.find_spec("spherely") is None:
    import sys

    try:
        import boto3

        s3 = boto3.client("s3")
        Path("wheels").mkdir(exist_ok=True)
        listing = s3.list_objects_v2(Bucket="sliderule-public", Prefix="zagg-demo/wheels/")
        for obj in listing["Contents"]:
            name = obj["Key"].rsplit("/", 1)[1]
            if name.endswith(".whl"):
                s3.download_file("sliderule-public", obj["Key"], f"wheels/{name}")
        !{sys.executable} -m pip install -q spherely --no-index --find-links wheels/
    except Exception as e:
        print(f"wheel fetch failed ({e}); continuing without spherely")

try:
    import spherely

    print("spherely", spherely.__version__, "| SpatialIndex:", hasattr(spherely, "SpatialIndex"))
except ImportError:
    print("spherely unavailable -- ShardMap.build will use the mortie backend")

spherely 0.1.1 | SpatialIndex: True


## 0 — Timing infra and auth

In [3]:
# prefer the 'nasa' profile when it exists (laptop); ambient creds otherwise (hub role)
import botocore.session as _bs

if "nasa" in (_bs.Session().full_config.get("profiles") or {}):
    os.environ.setdefault("AWS_PROFILE", "nasa")

logging.getLogger("stac_geoparquet").setLevel(logging.WARNING)

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
MISSION = ("2018-10-13", date.today().isoformat())

timings = {}

class stage:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        timings[self.name] = round(time.perf_counter() - self.t0, 2)
        print(f"[{self.name}] {timings[self.name]:.1f}s")


def healpix_grid(order):
    # ~10 m leaf cells (child order 19); footprint-intersection MOC at order 13 (#92)
    return HealpixGrid(order, child_order=19, chunk_inner=13)

## 1 — NEON SERC site

This is the site that we benchmark in CI/CD, and is the same AOI for most of the HHDC papers.

### 1a — NASA CMR: ATL03

In [4]:
serc = demo_aoi("serc")
grid9 = healpix_grid(9)
fetch_bbox = grid9.coverage_bbox(serc)  # shard-complete: cover the whole shards, not just the AOI

with stage("SERC: CMR query"):
    cat_serc = CMRSource().fetch(Query("ATL03", "007", *MISSION, region=serc))#fetch_bbox))
cat_serc.to_geoparquet(str(OUT / "catalog_atl03_serc.parquet"))
print(f"{len(cat_serc):,} granules")

[SERC: CMR query] 2.9s
68 granules


In [5]:
with stage("SERC: shardmap build (o9)"):
    sm_serc = ShardMap.build(cat_serc, grid9, region=load_polygon(serc), mortie_order=9)
sm_serc.to_json(str(OUT / "shardmap_atl03_serc_o9.json"))
{k: sm_serc.metadata[k] for k in ("total_shards", "total_granules", "granules_assigned", "total_pairs", "build_wall_s")}

[SERC: shardmap build (o9)] 0.0s


{'total_shards': 4,
 'total_granules': 68,
 'granules_assigned': 68,
 'total_pairs': 213,
 'build_wall_s': 0.009}

In [6]:
# AOI in green, shards in blue, icesat-2 geometries in red
show_shardmap(sm_serc, cat_serc, aoi=serc, zoom=10)

Map(center=[38.87384105897101, -76.552734375], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoo…

### 1b — STAC (Earth Search): Sentinel-2 L2A

Same area, different sensor

In [7]:
s2_source = STACSource(
    "https://earth-search.aws.element84.com/v1",
    assets=["red", "green", "blue", "nir", "scl"],
    time_key="s2:datatake_id",
)
s2_query = STACQuery(
    collections=["sentinel-2-c1-l2a", "sentinel-2-pre-c1-l2a"],
    start_date="2015-06-23",
    end_date=MISSION[1],
    region=serc, #fetch_bbox,
)
with stage("SERC: STAC query"):
    cat_s2 = s2_source.fetch(s2_query)
cat_s2.to_geoparquet(str(OUT / "catalog_s2_serc.parquet"))
datatakes = {r["time_key"] for r in cat_s2.granule_records()}
print(f"{len(cat_s2):,} items, {len(datatakes):,} datatakes")

[SERC: STAC query] 2.4s
528 items, 526 datatakes


In [8]:
s2_config = default_config("sentinel2_l2a")
s2_config.output["grid"]["parent_order"] = 9

with stage("SERC: S2 shardmap build (o9)"):
    sm_s2 = ShardMap.build(cat_s2, from_config(s2_config), region=load_polygon(serc), mortie_order=9)
sm_s2.to_json(str(OUT / "shardmap_s2_serc_o9.json"))
{k: sm_s2.metadata[k] for k in ("total_shards", "total_granules", "granules_assigned", "total_pairs", "build_wall_s")}

[SERC: S2 shardmap build (o9)] 0.1s


{'total_shards': 4,
 'total_granules': 528,
 'granules_assigned': 528,
 'total_pairs': 2111,
 'build_wall_s': 0.007}

In [9]:
show_shardmap(sm_s2, cat_s2, aoi=serc, zoom=9)

Map(center=[38.87384105897101, -76.552734375], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoo…

The plot above is instructive. Our query pipeline assumes a stac catalog; when we use the CMR or similar, we're just querying the STAC and returning the subset. Above, the STAC query returns coverage for the green box-- but zagg defines 4 shards that cover that box *and overhang it*. To ensure we fully fill all the cells, we'd need to wrap the AOI bounding box:

```python
fetch_bbox = grid9.coverage_bbox(serc)  # shard-complete: cover the whole shards, not just the AOI
```

Above would ensure all of the shards are complete. For an AOI that has a mask, maybe we don't care. For filling in and caching a map based on user requests, it might make sense to fully fill the shards with all of the data.

I'm calling this out on the Sentinel-2 plot, but the same thing happened for the ICESat-2 CMR query; if we rerun with fetch_bbox, we pick up more ICESat-2 granules there too.

## 2 — California, full mission: CMR vs a local catalog clone

Same CMR pipeline as SERC at state scale — then the same shard map built **straight off the local clone of the full ATL03 catalog** (555,867 granules, global), with no bbox prefilter: `ShardMap.build`'s exact footprint intersection (spherely / mortie) does all the spatial work. That's deliberate — it's the shape a global pipeline has, and bbox prefilters are exactly what bites at the poles (great-circle edge sag silently drops real granules). The bounding box survives only where CMR requires one, server-side.

~~Note that in contrast to the example above, hitting the local mission STAC catalog **always** is complete (assuming it's current with orbital cycles). That's because the shardmap geometry intersection happens on the full STAC catalog locally-- the reason the above example undersamples is because the AOI passes thru a separate 'pre-filter' query on the STAC API. When we have a local copy, we do the geometry intersection without any prefilter, and every shard in the shardmap references **all** the granules that pass through it.~~

edit: out of date, reverting to a bounding box prune for now.

In [10]:
ATL03_CLONE = Path("atl03_v007_full.parquet")
if not ATL03_CLONE.exists():
    with stage("fetch ATL03 catalog clone (305 MB)"):
        boto3.client("s3").download_file(
            "sliderule-public", "zagg-demo/atl03_v007_full.parquet", str(ATL03_CLONE)
        )

with stage("load ATL03 catalog clone"):
    full = Catalog.from_geoparquet(str(ATL03_CLONE))
print(f"{len(full):,} granules (full mission, global)")

[load ATL03 catalog clone] 0.1s
555,867 granules (full mission, global)


My cell above ran fast because the catalog was already cloned / copied to disk. The cell below hits CMR and runs slow for everyone, since we have to route over network for a large spatio-temporal query

In [11]:
ca = demo_aoi("california")

In [23]:
fetch_bbox_ca = healpix_grid(9).coverage_bbox(ca)  # CMR-side filter ONLY -- the local
                                                   # clone is never bbox-prefiltered

with stage("California: CMR query"):
    cat_ca_cmr = CMRSource().fetch(Query("ATL03", "007", *MISSION, region=fetch_bbox_ca))

print(f"CMR (exact footprint ∩ bbox, server-side): {len(cat_ca_cmr):,} granules "
      f"in {timings['California: CMR query']:.0f}s -- next, the local-clone path hands "
      f"ShardMap.build the FULL {len(full):,}-granule catalog and lets the exact "
      f"footprint intersection do all the spatial work")

[California: CMR query] 67.9s
CMR (exact footprint ∩ bbox, server-side): 2,686 granules in 68s -- next, the local-clone path hands ShardMap.build the FULL 555,867-granule catalog and lets the exact footprint intersection do all the spatial work


In [12]:
ca_parts = load_polygon(ca)
with stage("California: shardmap build (o9, full catalog)"):
    sm_ca9 = ShardMap.build(full, healpix_grid(9), region=ca_parts, mortie_order=9)
sm_ca9.to_json(str(OUT / "shardmap_california_o9.json"))

[California: shardmap build (o9, full catalog)] 267.0s


In [13]:
ca_parts = load_polygon(ca)
with stage("California: shardmap build (o9, full catalog)"):
    sm_ca9 = ShardMap.build(full, healpix_grid(9), region=ca_parts,
                            backend="mortie", mortie_order=9)
sm_ca9.to_json(str(OUT / "shardmap_california_o9_.json"))

[California: shardmap build (o9, full catalog)] 310.7s


In [21]:
%%time
sm_coarse = ShardMap.build(full, healpix_grid(1), region=ca_parts, backend="mortie", mortie_order=1)
survivors = {g["id"] for gl in sm_coarse.granules for g in gl}

CPU times: user 1min 23s, sys: 2min 56s, total: 4min 20s
Wall time: 1min 12s


In [14]:
%%time
sm_coarse = ShardMap.build(full, healpix_grid(5), region=ca_parts, backend="mortie", mortie_order=5)
survivors = {g["id"] for gl in sm_coarse.granules for g in gl}

CPU times: user 2min 18s, sys: 4min 16s, total: 6min 34s
Wall time: 1min 37s


In [17]:
import pyarrow.compute as pc
import pyarrow as pa

In [18]:
%%time
cat_ca = Catalog(full.table.filter(pc.is_in(full.table["id"], value_set=pa.array(survivors))), dict(full.metadata))
sm_ca9 = ShardMap.build(cat_ca, healpix_grid(9), region=ca_parts, mortie_order=9)

CPU times: user 7.13 s, sys: 2.51 s, total: 9.64 s
Wall time: 7.12 s


In [ ]:
# metadata total_granules = catalog records CONSIDERED (the whole clone); the exact
# footprint intersection assigns the distinct in-AOI subset:
assigned = {g["id"] for gl in sm_ca9.granules for g in gl}
print("exact intersection: {len(full):,} candidates -> {len(assigned):,} assigned "
      f"(CMR's footprint-exact count above: {len(cat_ca_cmr):,})")
{k: sm_ca9.metadata[k] for k in ("total_shards", "total_granules", "granules_assigned", "total_pairs", "build_wall_s")}

**The consolidated fast path** (the cells above are the working record; this is the keeper). Two stages, both sphere-correct footprint intersections — no bounding box anywhere: a coarse **exact** prefilter at o5, fanned out on a thread pool (mortie's Rust fill releases the GIL, so threads are real concurrency — and the o1-vs-o5 timings above show ~70s of the serial cost is per-granule call overhead, which is exactly what the threads hide), then the stock exact o9 build on the survivors. Assignment is byte-identical to the ~5-minute single-pass build. The native fix (batch coverage call / `workers=`) is [zagg#396](https://github.com/englacial/zagg/issues/396).

In [24]:
# ── fast two-stage build (tonight's workaround; native fix tracked in zagg#396) ──
from concurrent.futures import ThreadPoolExecutor

import pyarrow as pa
import pyarrow.compute as pc
from mortie import morton_coverage

COARSE = 1
ca_parts = load_polygon(ca)

# the AOI's o5 cells, once
aoi_cells = set()
for plats, plons in ca_parts:
    aoi_cells.update(int(c) for c in morton_coverage(plats, plons, order=COARSE))

def touches_aoi(rec):
    try:
        cells = morton_coverage(rec["lats"], rec["lons"], order=COARSE)
    except Exception:
        return None  # unfillable footprint: dropped, matching the exact build
    return rec["id"] if aoi_cells.intersection(int(c) for c in cells) else None

with stage("California: coarse prefilter (o5, threaded)"):
    with ThreadPoolExecutor(max_workers=8) as pool:
        survivors = sorted({gid for gid in pool.map(touches_aoi, full.granule_records()) if gid})

with stage("California: exact o9 build (survivors)"):
    cat_ca = Catalog(
        full.table.filter(pc.is_in(full.table["id"], value_set=pa.array(survivors))),
        dict(full.metadata),
    )
    sm_ca9 = ShardMap.build(cat_ca, healpix_grid(9), region=ca_parts, mortie_order=9)
sm_ca9.to_json(str(OUT / "shardmap_california_o9.json"))

assigned = {g["id"] for gl in sm_ca9.granules for g in gl}
print(f"{len(full):,} granules -> {len(survivors):,} survivors -> {len(assigned):,} assigned "
      f"(the single-pass exact build gives the identical set; CMR footprint-exact: {len(cat_ca_cmr):,})")
{k: sm_ca9.metadata[k] for k in ("total_shards", "total_granules", "granules_assigned", "total_pairs", "build_wall_s")}

[California: coarse prefilter (o5, threaded)] 44.6s
[California: exact o9 build (survivors)] 9.3s
555,867 granules -> 19,247 survivors -> 2,356 assigned (the single-pass exact build gives the identical set; CMR footprint-exact: 2,686)


{'total_shards': 2721,
 'total_granules': 19247,
 'granules_assigned': 2356,
 'total_pairs': 190625,
 'build_wall_s': 8.353}

Reprocess o9 → o8 in place: HEALPix nesting makes coarsening a pure regroup — no re-query, no geometry. (Caveat: shard-completeness is order-relative — o8 edge shards are wider than the o9 fetch box, so refetch with `healpix_grid(8).coverage_bbox(ca)` if the coarser store must stay shard-complete.)

In [13]:
with stage("California: reproject o9 -> o8"):
    sm_ca8 = sm_ca9.reproject(healpix_grid(8))
sm_ca8.to_json(str(OUT / "shardmap_california_o8.json"))
{**sm_ca8.metadata["reproject"], "total_shards": sm_ca8.metadata["total_shards"], "total_pairs": sm_ca8.metadata["total_pairs"]}

[California: reproject o9 -> o8] 0.1s


{'source_parent_order': 9,
 'target_parent_order': 8,
 'method': 'coarsen',
 'total_shards': 731,
 'total_pairs': 77310}

In [14]:
show_shardmap(sm_ca9, aoi=ca, zoom=6)

Map(center=[37.27664023406749, -119.267578125], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zo…

In [15]:
show_shardmap(sm_ca8, aoi=ca, zoom=6)

Map(center=[37.23830349410531, -119.267578125], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zo…

We use o9 shards with ICESat-2 for stability, but different missions might want different processing shard sizes.

## 3 — All NEON AOP domains (D01–D20)

61 flight-box parts, Alaska to Puerto Rico — the union bbox spans most of North America, so this is the local-clone path only: one shard-complete box per part.

This also show that we're quite tolerant of geometry inputs for zagg query-- multipart polygons and polygons with holes are both supported

In [16]:
neon_parts = load_polygon(demo_aoi("neon_aop"))
boxes = [healpix_grid(9).coverage_bbox([p]) for p in neon_parts]  # per part: one box per flight box

with stage("NEON: bbox prefilter"):
    cat_neon = full.filter_bbox(boxes)
print(f"{len(cat_neon):,} candidate granules over {len(neon_parts)} flight-box parts")

[NEON: bbox prefilter] 0.3s
22,116 candidate granules over 61 flight-box parts


In [18]:
with stage("NEON: shardmap build (o9)"):
    sm_neon = ShardMap.build(cat_neon, healpix_grid(9), region=neon_parts, mortie_order=9)
sm_neon.to_json(str(OUT / "shardmap_neon_aop_o9.json"))
{k: sm_neon.metadata[k] for k in ("total_shards", "total_granules", "granules_assigned", "total_pairs", "build_wall_s")}

[NEON: shardmap build (o9)] 2.6s


{'total_shards': 250,
 'total_granules': 22116,
 'granules_assigned': 5540,
 'total_pairs': 19999,
 'build_wall_s': 1.58}

In [19]:
show_shardmap(sm_neon, aoi=demo_aoi("neon_aop"), zoom=4)

Map(center=[44.686324250206326, -111.8839039522059], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## Timings

In [20]:
import json

import pandas as pd

(OUT / "timings_query.json").write_text(json.dumps(timings, indent=2))
pd.Series(timings, name="seconds").to_frame()

,seconds
SERC: CMR query,2.75
SERC: shardmap build (o9),0.02
SERC: STAC query,2.36
SERC: S2 shardmap build (o9),0.07
load ATL03 catalog clone,0.54
California: CMR query,70.02
California: local catalog cut,0.20
California: shardmap build (o9),6.78
California: reproject o9 -> o8,0.09
NEON: bbox prefilter,0.11
